### Loading data in hierarchal HDF5 format

Subject -> Stimulus Modality -> Stimulus Font -> Parity or Control

In [ ]:
import os
import pandas as pd
import mne
from pr_fe import FeatureExtractor  # adjust if needed
import h5py
import numpy as np

input_list_path = 'data/mat_files_cleaned.txt'
data_dir = 'data'
output_base_dir = 'hierarch_gr'

fe = FeatureExtractor()

def parse_filename(filename):
    base = os.path.basename(filename).replace('.mat', '')
    parts = base.split('_')

    subject = next((p[1:] for p in reversed(parts) if p.startswith('S')), 'Unknown')

    try:
        idx = parts.index('epbin') + 1
    except ValueError:
        idx = 1

    modality = parts[idx] if len(parts) > idx else 'UnknownModality'
    font = parts[idx + 1] if len(parts) > idx + 1 else 'UnknownFont'
    condition = parts[idx + 2] if len(parts) > idx + 2 else 'UnknownCondition'

    return subject, modality, font, condition

def load_and_prepare_csv(filepath):
    df = pd.read_csv(filepath)
    df_pivot = df.pivot_table(index='time', columns='channel', values='value')
    df_pivot.columns = df_pivot.columns.astype(str)
    return df_pivot, df

def main():
    with open(input_list_path, 'r') as f:
        files = [line.strip() for line in f if line.strip()]

    for file_rel_path in files:
        csv_rel_path = file_rel_path.replace('.mat', '.csv')
        file_path = os.path.join(data_dir, csv_rel_path)

        if not os.path.isfile(file_path):
            print(f"File not found: {file_path}, skipping.")
            continue

        print(f"Processing {file_path}...")

        subject, modality, font, condition = parse_filename(file_rel_path)
        df_signal, df_original = load_and_prepare_csv(file_path)

        ch_names = list(df_signal.columns.astype(str))
        ch_types = ['eeg'] * len(ch_names)
        sfreq = fe.sampling_rate
        data = df_signal.T.values

        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
        raw = mne.io.RawArray(data, info)

        features_df = fe.merging_feature_data(raw, df_signal)

        # --- Convert to numeric and keep only numeric columns to avoid HDF5 dtype issues ---
        features_df = features_df.apply(pd.to_numeric, errors='coerce')  # force non-numeric to NaN
        features_df = features_df.select_dtypes(include=[np.number])      # keep only numeric columns
        features_df = features_df.dropna(axis=1, how='all')               # drop columns that became all NaN

        output_dir = os.path.join(output_base_dir, f'S{subject}', modality, font)
        os.makedirs(output_dir, exist_ok=True)

        output_file = os.path.join(output_dir, f'{condition}.h5')
        group_name = f'S{subject}/{modality}/{font}'

        with h5py.File(output_file, 'a') as hdf5_file:
            group = hdf5_file.require_group(group_name)

            dataset_name = condition
            if dataset_name in group:
                print(f"Dataset {dataset_name} already exists in {group_name}, skipping.")
                continue

            data = features_df.to_numpy()
            group.create_dataset(dataset_name, data=data, chunks=True)
            group.attrs['columns'] = np.array(features_df.columns, dtype='S')

        print(f"Saved features to {output_file} under group {group_name}, dataset {dataset_name}")

if __name__ == '__main__':
    main()

Processing data/epbin_Dig_1F_C1_21_rr_fixAF7_fixF7_icfilt_ica_ep1_but_chanlocs_chansel_chanlabels_S19.csv...
Creating RawArray with float64 data, n_channels=70, n_times=30720
    Range : 0 ... 30719 =      0.000 ...    59.998 secs
Ready.
Effective window size : 0.500 (s)
Saved features to hierarch_gr/S19.csv/Dig/1F/C1.h5 under group S19.csv/Dig/1F, dataset C1
Processing data/epbin_Dig_1F_C1_21_rr_fixAF7_icfilt_ica_ep1_but_chanlocs_chansel_chanlabels_S09.csv...
Creating RawArray with float64 data, n_channels=70, n_times=30720
    Range : 0 ... 30719 =      0.000 ...    59.998 secs
Ready.
Effective window size : 0.500 (s)
Saved features to hierarch_gr/S09.csv/Dig/1F/C1.h5 under group S09.csv/Dig/1F, dataset C1
Processing data/epbin_Dig_1F_C1_21_rr_fixAF8_fixC5_fixFT7_ica_ep1_but_chanlocs_chansel_chanlabels_S15.csv...
Creating RawArray with float64 data, n_channels=70, n_times=30720
    Range : 0 ... 30719 =      0.000 ...    59.998 secs
Ready.
Effective window size : 0.500 (s)
Saved feat

KeyboardInterrupt: 